 * **validating and filtering content at key points in your agent’s execution**.
 * handling sensitive information, enforcing policies, output validation, unsafe behaviors
 * Using **middleware** **`langchain.agents.middleware`** - intercept execution at strategic points
 * **Deterministic Guardrails** - Rule based, Fast, cost effective, not very powerful
 * **Model Based Guardrails** - using llms/classifiers, slower, expensive, powerful

In [22]:
import asyncio
import os
from typing import Literal,TypedDict

## `PIIMiddleware`

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [32]:
from langchain_openai import ChatOpenAI
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model="gpt-4o",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("api_key",detector=r"sk-[a-zA-Z0-9]{32}",strategy="block",apply_to_input=True)
    ]
)

In [33]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

=== Agent Response ===
I found the customer record associated with your provided email. How can I assist you further with your account or inquiry?


In [52]:
def show_result(result):
    for i in result['messages']:
        print(type(i))
        try:
            print(i['content'])
        except:
            print(i)
        print('*'*100)

In [53]:
show_result(result)

<class 'langchain_core.messages.human.HumanMessage'>
content='\n        total sales = 13551, 12% improvement than q3.... this is Send an email to team@company.com about the Q4 results' additional_kwargs={} response_metadata={} id='3d5b01e7-aaef-44b5-a19d-f2cc790bdc6b'
****************************************************************************************************
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 128, 'total_tokens': 223, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_3c681aa847', 'id': 'chatcmpl-Dl03olbx10JNgNpcQwJrQ8h9aKrtr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} i

## `HumanInTheLoopMiddleware` 

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model="gpt-4o",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)


In [50]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [
        {"role": "user", "content": """
        total sales = 13551, 12% improvement than q3.... this is Send an email to team@company.com about the Q4 results"""}]
    },
    config=config
)

In [54]:
show_result(result)

<class 'langchain_core.messages.human.HumanMessage'>
content='\n        total sales = 13551, 12% improvement than q3.... this is Send an email to team@company.com about the Q4 results' additional_kwargs={} response_metadata={} id='3d5b01e7-aaef-44b5-a19d-f2cc790bdc6b'
****************************************************************************************************
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 128, 'total_tokens': 223, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_3c681aa847', 'id': 'chatcmpl-Dl03olbx10JNgNpcQwJrQ8h9aKrtr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} i

In [55]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

In [56]:
show_result(approved_result)

<class 'langchain_core.messages.human.HumanMessage'>
content='\n        total sales = 13551, 12% improvement than q3.... this is Send an email to team@company.com about the Q4 results' additional_kwargs={} response_metadata={} id='3d5b01e7-aaef-44b5-a19d-f2cc790bdc6b'
****************************************************************************************************
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 128, 'total_tokens': 223, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_3c681aa847', 'id': 'chatcmpl-Dl03olbx10JNgNpcQwJrQ8h9aKrtr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} i

In [57]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

In [58]:
show_result(rejected_result)

<class 'langchain_core.messages.human.HumanMessage'>
content='Delete all records from the users table where active=false' additional_kwargs={} response_metadata={} id='c8ba3fe4-4181-4465-8a2b-5b3714788728'
****************************************************************************************************
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 106, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_3c681aa847', 'id': 'chatcmpl-Dl0883092hg4cUl7cQ4Fs41hsMkGR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e75e3-3f43-7e80-8773-871054a63ca5-0' tool_calls=[

## `AgentMiddleware`
- creating custom middleware
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [59]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# Create agent with content filter
filtered_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

In [60]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print(result["messages"][-1].content)

Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to perform specific tasks without explicit instructions. Instead, these algorithms learn patterns and make decisions or predictions based on data. Machine learning is employed to analyze vast amounts of data, identify patterns, and make informed decisions or predictions.

There are several types of machine learning, including:

1. **Supervised Learning**: In this approach, the machine learning model is trained on a labeled dataset, which means that input and output data are already known. The model learns to map inputs to the correct outputs and can later use this knowledge to predict outcomes for new, unseen data. Examples include regression and classification tasks.

2. **Unsupervised Learning**: Here, the model is trained on an unlabeled dataset, and it attempts to identify patterns and relationships within the data. Clustering and associat

In [61]:
# Test 2: UnSafe request — should not pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What can I hack a server?"}]
})
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
I cannot process requests containing inappropriate content. Please rephrase your request.


In [65]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """
    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response to evaluate:
        {last_message.content}
        """
        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])
        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )
        return None

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

safe_agent = create_agent(
    model="gpt-4o",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

In [66]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today in India?"}]
})
show_result(result)

<class 'langchain_core.messages.human.HumanMessage'>
content='What is the weather like today in India?' additional_kwargs={} response_metadata={} id='4be1280a-4de3-4c9d-8a4c-735a792e1a51'
****************************************************************************************************
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 51, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fa0699a156', 'id': 'chatcmpl-Dl0R6srG7PEz3dqhbRNKJLArrrjYi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e75f5-3334-7153-9767-fd940d687738-0' tool_calls=[{'name': 'general_to

## Combined Guardrails

In [68]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)